In [1]:
import pandas as pd
import numpy as np
import random

# Set random seed for reproducibility
np.random.seed(42)

# Number of rows (users) to generate
num_users = 100

# Helper function to generate random data
def generate_user_data(num_users):
    user_data = []
    
    for user_id in range(1, num_users + 1):
        # Generate random user attributes
        age = random.randint(18, 60)
        gender = random.choice(['Male', 'Female', 'Other'])
        location = random.choice(['Bangalore', 'Pune', 'Nagpur', 'Surat', 'Visakhapatnam'])
        income_level = random.choice(['Low', 'Medium', 'High'])
        preferred_categories = random.sample(['Electronics', 'Grocery', 'Fashion', 'Home Essentials', 'Stationery'], 3)
        budget_range = f"{random.randint(1000, 10000)}-{random.randint(10000, 20000)}"
        brand_preference = random.choice(['Apple', 'Samsung', 'Sony', 'LG', 'None'])
        frequency_of_purchase = random.randint(1, 10)  # times per month
        discount_sensitivity = random.choice(['High', 'Medium', 'Low'])
        past_purchases = random.sample(['Smartphone', 'Laptop', 'Headphones', 'TV', 'Fridge'], 3)
        wishlist = random.sample(['Smartwatch', 'Camera', 'Smartphone Accessories'], 2)
        engagement_score = random.randint(1, 10)
        satisfaction_rating = round(random.uniform(3.0, 5.0), 1)
        ratings = [random.randint(1, 5) for _ in range(4)]  # 4 ratings for products

        # Append the user data to the list
        user_data.append([
            user_id, age, gender, location, income_level, 
            ', '.join(preferred_categories), budget_range, brand_preference,
            frequency_of_purchase, discount_sensitivity, ', '.join(past_purchases), 
            ', '.join(wishlist), engagement_score, satisfaction_rating, ratings
        ])
    
    # Create DataFrame from the generated data
    columns = [
        'User_ID', 'Age', 'Gender', 'Location', 'Income_Level',
        'Preferred_Categories', 'Budget_Range', 'Brand_Preference',
        'Frequency_of_Purchase', 'Discount_Sensitivity', 'Past_Purchases', 
        'Wishlist', 'Engagement_Score', 'Satisfaction_Rating', 'Ratings'
    ]
    
    return pd.DataFrame(user_data, columns=columns)

# Generate the dummy user dataset
user_df = generate_user_data(num_users)

# Save the DataFrame to a CSV file
user_df.to_csv('data/user_data.csv', index=False)

print("Dummy user dataset has been generated and saved to 'data/user_data.csv'.")


Dummy user dataset has been generated and saved to 'data/user_data.csv'.


In [6]:
df=pd.read_csv('data/user_data.csv')
df.head()

,User_ID,Age,Gender,Location,Income_Level,Preferred_Categories,Budget_Range,Brand_Preference,Frequency_of_Purchase,Discount_Sensitivity,Past_Purchases,Wishlist,Engagement_Score,Satisfaction_Rating,Ratings
0,1,49,Female,Visakhapatnam,High,"Electronics, Stationery, Home Essentials",7419-17923,LG,8,Low,"Laptop, Smartphone, TV","Smartphone Accessories, Camera",10,3.9,"[2, 5, 5, 1]"
1,2,36,Other,Surat,Low,"Fashion, Grocery, Home Essentials",3206-11265,LG,10,Low,"Smartphone, TV, Headphones","Smartphone Accessories, Smartwatch",10,3.7,"[4, 5, 4, 5]"
2,3,57,Male,Visakhapatnam,Low,"Home Essentials, Electronics, Grocery",8906-17630,Sony,10,Low,"TV, Fridge, Laptop","Smartphone Accessories, Camera",4,4.0,"[2, 2, 1, 2]"
3,4,48,Male,Nagpur,Medium,"Fashion, Stationery, Home Essentials",4939-11937,Samsung,6,High,"Fridge, Smartphone, Headphones","Smartwatch, Camera",1,4.8,"[2, 1, 3, 3]"
4,5,41,Female,Surat,High,"Electronics, Grocery, Fashion",8143-13802,Sony,5,Low,"Fridge, Laptop, Smartphone","Camera, Smartphone Accessories",3,3.7,"[1, 4, 5, 4]"


In [11]:
# Convert categorical columns into lists for easier processing
user_df['Preferred_Categories'] = user_df['Preferred_Categories'].apply(lambda x: x.split(', '))
user_df['Past_Purchases'] = user_df['Past_Purchases'].apply(lambda x: x.split(', '))
user_df['Wishlist'] = user_df['Wishlist'].apply(lambda x: x.split(', '))

# Create a combined feature for user preferences
user_df['User_Preferences'] = user_df['Preferred_Categories'] + user_df['Past_Purchases'] + user_df['Wishlist']

# Display the updated DataFrame
print(user_df[['User_ID', 'User_Preferences']].head())

   User_ID                                   User_Preferences
0        1  [Electronics, Stationery, Home Essentials, Lap...
1        2  [Fashion, Grocery, Home Essentials, Smartphone...
2        3  [Home Essentials, Electronics, Grocery, TV, Fr...
3        4  [Fashion, Stationery, Home Essentials, Fridge,...
4        5  [Electronics, Grocery, Fashion, Fridge, Laptop...


In [12]:
from sklearn.preprocessing import MultiLabelBinarizer

# Flatten the User_Preferences column into a list of items
mlb = MultiLabelBinarizer()
user_item_matrix = pd.DataFrame(mlb.fit_transform(user_df['User_Preferences']), columns=mlb.classes_, index=user_df['User_ID'])

# Display the user-item matrix
print(user_item_matrix.head())

         Camera  Electronics  Fashion  Fridge  Grocery  Headphones  \
User_ID                                                              
1             1            1        0       0        0           0   
2             0            0        1       0        1           1   
3             1            1        0       1        1           0   
4             1            0        1       1        0           1   
5             1            1        1       1        1           0   

         Home Essentials  Laptop  Smartphone  Smartphone Accessories  \
User_ID                                                                
1                      1       1           1                       1   
2                      1       0           1                       1   
3                      1       1           0                       1   
4                      1       0           1                       0   
5                      0       1           1                       1   

    

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute user-user similarity
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

# Display user similarity matrix
print(user_similarity_df.head())

User_ID    1      2      3      4      5      6      7      8      9      10   \
User_ID                                                                         
1        1.000  0.500  0.750  0.500  0.625  0.500  0.625  0.625  0.625  0.750   
2        0.500  1.000  0.500  0.625  0.500  0.625  0.625  0.500  0.625  0.625   
3        0.750  0.500  1.000  0.375  0.750  0.375  0.750  0.500  0.625  0.625   
4        0.500  0.625  0.375  1.000  0.500  0.750  0.625  0.625  0.625  0.500   
5        0.625  0.500  0.750  0.500  1.000  0.625  0.750  0.625  0.625  0.750   

User_ID  ...    91     92     93     94     95     96     97     98     99   \
User_ID  ...                                                                  
1        ...  0.625  0.750  0.625  0.500  0.625  0.875  0.750  0.625  0.500   
2        ...  0.500  0.625  0.750  0.625  0.750  0.500  0.500  0.375  0.750   
3        ...  0.750  0.625  0.625  0.500  0.500  0.625  0.875  0.500  0.625   
4        ...  0.625  0.500  0.500  0.

In [17]:
def recommend_items(user_id, top_n=5):
    # Get the similarity scores for the user
    user_scores = user_similarity_df.loc[user_id]

    # Sort users by similarity (excluding the user themselves)
    similar_users = user_scores.drop(user_id).sort_values(ascending=False)

    # Get items interacted with by similar users
    similar_users_items = user_item_matrix.loc[similar_users.index]
    recommended_items = similar_users_items.sum().sort_values(ascending=False)

    # Exclude items the user has already interacted with
    user_items = user_item_matrix.loc[user_id]
    recommended_items = recommended_items.drop(user_items[user_items > 0].index)

    # Return the top N recommended items
    return recommended_items.head(top_n).index.tolist()

# Example: Get recommendations for user 1
print(recommend_items(1, top_n=5))

['Fashion', 'Fridge', 'Smartwatch', 'Headphones', 'Grocery']


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Example: Use User_Preferences as item features
item_features = user_df['User_Preferences'].apply(lambda x: ' '.join(x))

# Convert item features into TF-IDF vectors
vectorizer = TfidfVectorizer()
item_vectors = vectorizer.fit_transform(item_features)

# Display item feature vectors
print(item_vectors.shape)

(100, 14)


In [24]:
# Get the unique items from the user-item matrix
unique_items = user_item_matrix.columns

# Filter the User_Preferences to include only the unique items in the user-item matrix
filtered_preferences = user_df['User_Preferences'].apply(lambda x: [item for item in x if item in unique_items])

# Convert filtered preferences into TF-IDF vectors
vectorizer = TfidfVectorizer()
item_vectors = vectorizer.fit_transform(filtered_preferences.apply(lambda x: ' '.join(x)))

# Verify the shapes
print("Shape of item_vectors:", item_vectors.shape)
print("Number of unique items in user_item_matrix:", len(unique_items))

Shape of item_vectors: (100, 14)
Number of unique items in user_item_matrix: 13


In [25]:
# Get the unique items from the user-item matrix
unique_items = user_item_matrix.columns

# Get the unique items from the item_vectors matrix
item_names = vectorizer.get_feature_names_out()

# Find the extra item
extra_item = set(item_names) - set(unique_items)
print("Extra item:", extra_item)

Extra item: {'home', 'stationery', 'fridge', 'accessories', 'grocery', 'camera', 'fashion', 'tv', 'headphones', 'smartwatch', 'essentials', 'smartphone', 'laptop', 'electronics'}


In [28]:
from scipy.sparse import csr_matrix
# Remove the extra item from item_vectors
# Remove the extra item from item_vectors
if extra_item:
    extra_item_index = list(item_names).index(extra_item.pop())
    item_vectors = np.delete(item_vectors, extra_item_index, axis=1)  # No need for .toarray()
    item_vectors = csr_matrix(item_vectors)  # Convert back to sparse matrix
    item_names = np.delete(item_names, extra_item_index)

# Verify the shapes
print("Shape of item_vectors after filtering:", item_vectors.shape)
print("Number of unique items in user_item_matrix:", len(unique_items))

Shape of item_vectors after filtering: (100, 12)
Number of unique items in user_item_matrix: 13


In [29]:
# Get the unique items from the user-item matrix
unique_items = user_item_matrix.columns

# Get the unique items from the item_vectors matrix
item_names = vectorizer.get_feature_names_out()

# Find the extra item
extra_item = set(item_names) - set(unique_items)
print("Extra item:", extra_item)

Extra item: {'home', 'stationery', 'fridge', 'accessories', 'grocery', 'camera', 'fashion', 'tv', 'headphones', 'smartwatch', 'essentials', 'smartphone', 'laptop', 'electronics'}


In [30]:
# Get the unique items from the user-item matrix
unique_items = user_item_matrix.columns

# Get the unique items from the item_vectors matrix
item_names = vectorizer.get_feature_names_out()

print("Unique items in user_item_matrix:", unique_items)
print("Unique items in item_vectors:", item_names)

Unique items in user_item_matrix: Index(['Camera', 'Electronics', 'Fashion', 'Fridge', 'Grocery', 'Headphones',
       'Home Essentials', 'Laptop', 'Smartphone', 'Smartphone Accessories',
       'Smartwatch', 'Stationery', 'TV'],
      dtype='object')
Unique items in item_vectors: ['accessories' 'camera' 'electronics' 'essentials' 'fashion' 'fridge'
 'grocery' 'headphones' 'home' 'laptop' 'smartphone' 'smartwatch'
 'stationery' 'tv']


In [31]:
# Find items in item_names but not in unique_items
extra_items = set(item_names) - set(unique_items)
print("Extra items in item_vectors:", extra_items)

# Find items in unique_items but not in item_names
missing_items = set(unique_items) - set(item_names)
print("Missing items in item_vectors:", missing_items)

Extra items in item_vectors: {'home', 'stationery', 'fridge', 'accessories', 'grocery', 'camera', 'fashion', 'tv', 'headphones', 'smartwatch', 'essentials', 'smartphone', 'laptop', 'electronics'}
Missing items in item_vectors: {'TV', 'Camera', 'Electronics', 'Fridge', 'Headphones', 'Stationery', 'Home Essentials', 'Laptop', 'Fashion', 'Smartphone', 'Smartwatch', 'Grocery', 'Smartphone Accessories'}


In [32]:
# Standardize item names in user_item_matrix
user_item_matrix.columns = [item.lower().replace(' ', '_') for item in user_item_matrix.columns]

# Standardize item names in User_Preferences
user_df['User_Preferences'] = user_df['User_Preferences'].apply(
    lambda x: [item.lower().replace(' ', '_') for item in x]
)

# Verify the unique items in user_item_matrix
unique_items = user_item_matrix.columns
print("Unique items in user_item_matrix:", unique_items)

Unique items in user_item_matrix: Index(['camera', 'electronics', 'fashion', 'fridge', 'grocery', 'headphones',
       'home_essentials', 'laptop', 'smartphone', 'smartphone_accessories',
       'smartwatch', 'stationery', 'tv'],
      dtype='object')


In [33]:
# Convert filtered preferences into TF-IDF vectors
vectorizer = TfidfVectorizer()
item_vectors = vectorizer.fit_transform(user_df['User_Preferences'].apply(lambda x: ' '.join(x)))

# Get the unique items from the item_vectors matrix
item_names = vectorizer.get_feature_names_out()
print("Unique items in item_vectors:", item_names)

Unique items in item_vectors: ['camera' 'electronics' 'fashion' 'fridge' 'grocery' 'headphones'
 'home_essentials' 'laptop' 'smartphone' 'smartphone_accessories'
 'smartwatch' 'stationery' 'tv']


In [34]:
# Check if item_names and unique_items are aligned
if set(item_names) != set(unique_items):
    print("Mismatch between item_names and unique_items.")
    print("Items in item_names but not in unique_items:", set(item_names) - set(unique_items))
    print("Items in unique_items but not in item_names:", set(unique_items) - set(item_names))
else:
    print("Item names are aligned!")

Item names are aligned!


In [37]:
df=pd.read_csv('data/user_data.csv')
df.head()

,User_ID,Age,Gender,Location,Income_Level,Preferred_Categories,Budget_Range,Brand_Preference,Frequency_of_Purchase,Discount_Sensitivity,Past_Purchases,Wishlist,Engagement_Score,Satisfaction_Rating,Ratings
0,1,49,Female,Visakhapatnam,High,"Electronics, Stationery, Home Essentials",7419-17923,LG,8,Low,"Laptop, Smartphone, TV","Smartphone Accessories, Camera",10,3.9,"[2, 5, 5, 1]"
1,2,36,Other,Surat,Low,"Fashion, Grocery, Home Essentials",3206-11265,LG,10,Low,"Smartphone, TV, Headphones","Smartphone Accessories, Smartwatch",10,3.7,"[4, 5, 4, 5]"
2,3,57,Male,Visakhapatnam,Low,"Home Essentials, Electronics, Grocery",8906-17630,Sony,10,Low,"TV, Fridge, Laptop","Smartphone Accessories, Camera",4,4.0,"[2, 2, 1, 2]"
3,4,48,Male,Nagpur,Medium,"Fashion, Stationery, Home Essentials",4939-11937,Samsung,6,High,"Fridge, Smartphone, Headphones","Smartwatch, Camera",1,4.8,"[2, 1, 3, 3]"
4,5,41,Female,Surat,High,"Electronics, Grocery, Fashion",8143-13802,Sony,5,Low,"Fridge, Laptop, Smartphone","Camera, Smartphone Accessories",3,3.7,"[1, 4, 5, 4]"


In [38]:
import pandas as pd

# Create a user-item interaction matrix
user_item_matrix = pd.get_dummies(user_df['User_Preferences'].explode()).groupby(level=0).sum()

In [39]:
import numpy as np

# Perform matrix factorization using SVD
U, sigma, Vt = np.linalg.svd(user_item_matrix, full_matrices=False)

In [40]:
# Reconstruct the matrix with reduced dimensions
k = 5  # Number of latent factors
user_factors = U[:, :k]
item_factors = Vt[:k, :]

In [41]:
# Predict user-item interactions
predicted_ratings = np.dot(user_factors, item_factors)

In [42]:
# Convert predictions to a DataFrame
predicted_ratings_df = pd.DataFrame(predicted_ratings, index=user_item_matrix.index, columns=user_item_matrix.columns)

# Display predicted ratings
print(predicted_ratings_df.head())

     camera  electronics   fashion    fridge   grocery  headphones  \
0  0.042689     0.009187  0.025734 -0.000019  0.015963   -0.063764   
1 -0.097382     0.013851  0.018241 -0.098116  0.091528    0.061328   
2  0.107476     0.027317 -0.063211  0.081921  0.077706   -0.068266   
3  0.055025     0.024842  0.030195  0.074929  0.004052    0.076149   
4  0.069520     0.080814  0.041057  0.072058  0.026203    0.013957   

   home_essentials    laptop  smartphone  smartphone_accessories  smartwatch  \
0         0.075118  0.087186    0.008467                0.095534   -0.044755   
1         0.060967  0.021095    0.117891                0.105972    0.086416   
2         0.134445  0.047103   -0.007625                0.083721   -0.100883   
3        -0.002870  0.011972    0.019107               -0.073410    0.105511   
4        -0.004604 -0.024410    0.035508                0.095387   -0.073870   

   stationery        tv  
0    0.014200  0.108331  
1   -0.042077  0.040312  
2   -0.040786  0.082

In [43]:
def recommend_items(user_id, top_n=5):
    # Get the predicted ratings for the user
    user_ratings = predicted_ratings_df.loc[user_id]

    # Sort items by predicted rating
    recommended_items = user_ratings.sort_values(ascending=False).index.tolist()

    # Return the top N recommended items
    return recommended_items[:top_n]

# Example: Get recommendations for user 1
print(recommend_items(1, top_n=5))

['smartphone', 'smartphone_accessories', 'grocery', 'smartwatch', 'headphones']


In [46]:
from sklearn.model_selection import train_test_split

# Split the user-item matrix into training and testing sets
train_matrix, test_matrix = train_test_split(user_item_matrix, test_size=0.2, random_state=42)

# Get the indices of the test users
test_user_indices = test_matrix.index

# Extract the user factors for the test users
test_user_factors = user_factors[test_user_indices - 1]  # Adjust for 0-based indexing

In [47]:
# Predict ratings for the test set
test_predictions = np.dot(test_user_factors, item_factors)

In [48]:
from sklearn.metrics import mean_squared_error

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(test_matrix, test_predictions))
print("RMSE:", rmse)

RMSE: 0.7637534062525471


In [52]:
def recommend_items(user_id, top_n=5):
    # Get the predicted ratings for the user
    user_ratings = np.dot(user_factors[user_id - 1], item_factors)

    # Sort items by predicted rating
    recommended_items = user_item_matrix.columns[np.argsort(-user_ratings)[:top_n]].tolist()

    return recommended_items

# Example: Get recommendations for user 1
print(recommend_items(1, top_n=5))

['stationery', 'smartphone', 'electronics', 'tv', 'home_essentials']


In [53]:
# Save the user-item matrix to a CSV file
user_item_matrix.to_csv('user_item_matrix.csv', index=True)

In [54]:
user_item_matrix = pd.read_csv('user_item_matrix.csv')
print(user_item_matrix.columns)


Index(['Unnamed: 0', 'camera', 'electronics', 'fashion', 'fridge', 'grocery',
       'headphones', 'home_essentials', 'laptop', 'smartphone',
       'smartphone_accessories', 'smartwatch', 'stationery', 'tv'],
      dtype='object')


In [55]:
# Save the user-item matrix to a CSV file with a named index
user_item_matrix.to_csv('user_item_matrix.csv', index=True, index_label='User_ID')

In [56]:
user_item_matrix = pd.read_csv('user_item_matrix.csv', index_col='User_ID')
print(user_item_matrix.head())

         Unnamed: 0  camera  electronics  fashion  fridge  grocery  \
User_ID                                                              
0                 0       1            1        0       0        0   
1                 1       0            0        1       0        1   
2                 2       1            1        0       1        1   
3                 3       1            0        1       1        0   
4                 4       1            1        1       1        1   

         headphones  home_essentials  laptop  smartphone  \
User_ID                                                    
0                 0                1       1           1   
1                 1                1       0           1   
2                 0                1       1           0   
3                 1                1       0           1   
4                 0                0       1           1   

         smartphone_accessories  smartwatch  stationery  tv  
User_ID                   

In [57]:
# Save the user-item matrix to a CSV file with a named index
user_item_matrix.to_csv('user_item_matrix.csv', index=True, index_label='User_ID')